# RNA–ATAC dynamics in kidney organoids

The GSE213152 dataset describes kidney organoid development through paired RNA and chromatin-accessibility measurements. MOFI links the two modalities to reconstruct trajectories, cell-fate composition and marker dynamics (Figure 3).


In [ ]:
from pathlib import Path
import json

from mofi_notebook import setup_notebook, suppress_live_plot_output

ROOT, DEVICE, _ = setup_notebook()
print(f"Device: {DEVICE}")


## Data and fitted models

The analysis uses processed RNA and ATAC data, a fitted dynamics model and a cross-modality map.


In [ ]:
from CytoBridge.reproducibility import require_paper_data

DATASET = "gse213152"
paths = require_paper_data(DATASET, ROOT)
paths


## Continuous-time dynamics

The fitted model interpolates cell states between the observed time points.


In [ ]:
from CytoBridge.reproducibility import run_real_downstream

paper_output = ROOT / "results" / "figure_3_paper"
with suppress_live_plot_output():
    downstream = run_real_downstream(
        DATASET,
        output_dir=paper_output,
        selected_figures=["cell_type_main"],
        device=DEVICE,
        sample_max_cells=None,
        reuse_cache=True,
        repo_root=ROOT,
    )
run_dir = Path(downstream["run_dir"])
run_dir


## Synchronized RNA and ATAC trajectories

Trajectories in the two modalities share the same starting cells and time grid, allowing transcriptional and chromatin changes to be compared.


In [ ]:
from CytoBridge.manuscript import render_synchronized_corridor

corridor_dir = run_dir / "figures" / "synchronized_corridor"
with suppress_live_plot_output():
    render_synchronized_corridor(
        DATASET,
        output_dir=corridor_dir,
        device=DEVICE,
        direction="1to2",
        repo_root=ROOT,
    )
corridor_dir


In [ ]:
from mofi_notebook import show_paper_panel
show_paper_panel(corridor_dir / "sync_multimodal_time_corridor.png")


## Cell-fate dynamics

Predicted cell-type proportions describe population changes along the reconstructed process.


In [ ]:
from mofi_notebook import show_paper_panel
show_paper_panel(run_dir / "figures" / "cell_type_fate_dynamics.png")


## Marker dynamics

Model predictions are compared with observed time-point means for RNA and ATAC-linked markers.


In [ ]:
from CytoBridge.manuscript import plot_marker_bridging

with suppress_live_plot_output():
    marker_outputs = plot_marker_bridging(
        DATASET,
        run_dir=run_dir,
        output_dir=run_dir / "figures" / "manuscript_markers",
        markers_by_modality={
            "main_var": ["HNF1B", "PAX8"],
            "sec_var": ["EYA1", "PAX8"],
        },
        config_output_dir=paper_output,
        device=DEVICE,
        repo_root=ROOT,
    )
marker_outputs


### RNA markers: HNF1B and PAX8


In [ ]:
from mofi_notebook import show_paper_panel
show_paper_panel(run_dir / "figures" / "manuscript_markers" / "HNF1B_PAX8.png")


### ATAC-linked markers: EYA1 and PAX8


In [ ]:
from mofi_notebook import show_paper_panel
show_paper_panel(run_dir / "figures" / "manuscript_markers" / "EYA1_PAX8.png")


## Trajectories and growth

The inferred trajectories and growth rates are shown in the modality-specific UMAP embeddings.

The RNA and ATAC trajectory panels use different historical fitted dynamics models: the RNA panel uses the balanced-cell model, while the ATAC panel uses the full-data model. The starting indices and projections are specified in the analysis code.


In [ ]:
from CytoBridge.reproducibility import run_paper_ode_v3

with suppress_live_plot_output():
    ode_v3 = run_paper_ode_v3(
        DATASET,
        output_dir=run_dir / "figures" / "paper_ode_v3",
        device=DEVICE,
        repo_root=ROOT,
    )
ode_v3["outputs"]


### RNA trajectories


In [ ]:
from mofi_notebook import show_paper_panel
show_paper_panel(Path(ode_v3["outputs"]["primary_trajectory_png"]))


### ATAC trajectories


In [ ]:
from mofi_notebook import show_paper_panel
show_paper_panel(Path(ode_v3["outputs"]["secondary_trajectory_png"]))


### Growth in RNA space


In [ ]:
from mofi_notebook import show_paper_panel
show_paper_panel(Path(ode_v3["outputs"]["primary_growth_png"]))


### Growth in ATAC space


In [ ]:
from mofi_notebook import show_paper_panel
show_paper_panel(Path(ode_v3["outputs"]["secondary_growth_png"]))
